<a href="https://colab.research.google.com/github/sruthi-kurra/llm-distillation/blob/main/01_teacher_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1 — Install dependencies

In [1]:
!pip install transformers datasets evaluate rouge_score sentencepiece accelerate -q

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import evaluate

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("All dependencies installed!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB
All dependencies installed!


## Cell 2 — Load BART Large CNN teacher model

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading BART Large CNN teacher model...")
teacher = AutoModelForSeq2SeqLM.from_pretrained(model_name)
teacher = teacher.cuda()
teacher.eval()

total_params = sum(p.numel() for p in teacher.parameters())
print(f"\nTeacher model: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Model size: {total_params / 1e6:.1f}M parameters")
print(f"Device: {next(teacher.parameters()).device}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading tokenizer...


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]


Teacher model: facebook/bart-large-cnn
Total parameters: 406,290,432
Model size: 406.3M parameters
Device: cuda:0
GPU memory allocated: 1.63 GB


## Cell 3 — Load CNN/DailyMail dataset (20k subset)

In [5]:
from datasets import load_dataset

print("Loading CNN/DailyMail dataset...")
full_dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

# Take subsets for faster iteration
train_dataset = full_dataset['train'].select(range(20000))
val_dataset   = full_dataset['validation'].select(range(1000))
test_dataset  = full_dataset['test'].select(range(500))

print(f"\nDataset subsets:")
print(f"Train: {len(train_dataset):,} articles")
print(f"Validation: {len(val_dataset):,} articles")
print(f"Test: {len(test_dataset):,} articles")

# Preview one example
example = test_dataset[0]
print(f"\nExample article (first 300 chars):")
print(example['article'][:300])
print(f"\nReference summary:")
print(example['highlights'])

Loading CNN/DailyMail dataset...

Dataset subsets:
Train: 20,000 articles
Validation: 1,000 articles
Test: 500 articles

Example article (first 300 chars):
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the cou

Reference summary:
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


## Cell 4 — Evaluate teacher ROUGE baseline

In [7]:
import evaluate
import torch

rouge = evaluate.load("rouge")

def generate_summary(article, max_input_length=512, max_output_length=128):
    inputs = tokenizer(
        article,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        summary_ids = teacher.generate(
            inputs["input_ids"],
            num_beams=4,
            max_length=max_output_length,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Evaluating teacher on 100 test examples...")
predictions = []
references  = []

eval_subset = test_dataset.select(range(100))

for i, example in enumerate(eval_subset):
    pred = generate_summary(example['article'])
    predictions.append(pred)
    references.append(example['highlights'])
    if (i+1) % 20 == 0:
        print(f"Processed {i+1}/100...")

results = rouge.compute(predictions=predictions, references=references)

print(f"\n=== Teacher (BART Large CNN) ROUGE Scores ===")
print(f"ROUGE-1: {results['rouge1']:.4f}")
print(f"ROUGE-2: {results['rouge2']:.4f}")
print(f"ROUGE-L: {results['rougeL']:.4f}")

teacher_rouge = results
print("\nTeacher baseline saved!")

Evaluating teacher on 100 test examples...
Processed 20/100...
Processed 40/100...
Processed 60/100...
Processed 80/100...
Processed 100/100...

=== Teacher (BART Large CNN) ROUGE Scores ===
ROUGE-1: 0.3394
ROUGE-2: 0.1497
ROUGE-L: 0.2555

Teacher baseline saved!
